In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# 1. Install required libraries
!pip install -q transformers peft bitsandbytes accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# 2. Define our paths
base_model_name = "Qwen/Qwen2.5-3B"

# !!! UPDATE THIS PATH to point to your uploaded Kaggle dataset folder !!!
# It should be the folder that contains adapter_config.json and adapter_model.safetensors
adapter_path = "/kaggle/input/models/shehnazzzzazwala/recipe-generator/pytorch/default/1"

# 3. Setup 4-bit Quantization (So the giant base model fits on Kaggle)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("⏳ Loading the giant base Qwen model (This takes a minute)...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("🧩 Attaching your custom Recipe Adapter...")
# This is the magic line that merges your training with Qwen's brain
model = PeftModel.from_pretrained(base_model, adapter_path)

print("✅ Model successfully loaded and merged!\n")
print("="*50)

# 4. The AI Chef Generation Function
def generate_recipe(title, ingredients):
    # We MUST format this exactly how your cleaned dataset was formatted!
    prompt = f"Title: {title} Ingredients: {ingredients} \n\nRecipe: "
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    print(f"👨‍🍳 AI Chef Qwen is cooking '{title}'...\n")
    
    # Generate the text
    with torch.no_grad(): # Tells PyTorch we are just testing, not training (saves memory)
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,       # Max length of the recipe
            temperature=0.7,          # 0.7 gives a great balance of logic and creativity
            top_p=0.9,
            repetition_penalty=1.15,  # High penalty to stop loops
            do_sample=True,       # Stops it from repeating "stir, stir, stir"
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode the output
    # Decode the output
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    recipe_only = full_output.split("Recipe: ")[-1].strip()
    
    # --- THE ANTI-BLOGGER GUARDRAIL ---
    # We tell Python to instantly chop off the text if it spots any of these common blog/legal phrases
    spam_triggers = [
        "This recipe is made available", 
        "Please note that", 
        "If you like this recipe", 
        "Follow us on", 
        "Check out our", 
        "You can also make this recipe ahead",
        "Happy grilling",
        "Disclaimer"
    ]
    
    clean_recipe = recipe_only
    for trigger in spam_triggers:
        if trigger in clean_recipe:
            clean_recipe = clean_recipe.split(trigger)[0].strip()
            
    # Sometimes it rambles after "Enjoy!", so let's cut it there but keep the word "Enjoy!"
    if "Enjoy!" in clean_recipe:
        clean_recipe = clean_recipe.split("Enjoy!")[0] + "Enjoy!"
        
    print(clean_recipe)
    print("\n" + "="*50 + "\n")

# --- TEST YOUR MODEL HERE ---

# Test 1: Let's see if it finally knows how to use Garlic and Butter!
generate_recipe(
    title="Garlic Butter Lemon Chicken", 
    ingredients="chicken breasts, butter, minced garlic, lemon juice, salt, black pepper, parsley"
)

# Test 2: Let's give it something weird to see its creativity
generate_recipe(
    title="Spicy Mango Volcano Tacos", 
    ingredients="ground beef, taco shells, diced mango, jalapenos, hot sauce, cheddar cheese"
)

In [ ]:
# 1. Install required libraries

!pip install -q -U gspread oauth2client bitsandbytes>=0.46.1 transformers==4.43.0 peft accelerate gradio
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from datetime import datetime

# 2. Define our paths
base_model_name = "Qwen/Qwen2.5-3B"
adapter_path = "/kaggle/input/models/shehnazzzzazwala/recipe-generator/pytorch/default/1"

# 3. Setup 4-bit Quantization and Load Models
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("⏳ Loading the AI Chef...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, adapter_path)
print("✅ Kitchen is open!\n")
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# 1. Setup the connection
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
# IMPORTANT: Upload your JSON key to Kaggle and put the path here
creds = ServiceAccountCredentials.from_json_keyfile_name('/kaggle/input/datasets/shehnazzzzazwala/google-creds/positive-tempo-491407-i4-b1947ea458e2.json', scope)
client = gspread.authorize(creds)

# 2. Open the sheet
sheet = client.open("AI_Chef_History").sheet1

def save_to_cloud(title, recipe):
    try:
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
        # We use insert_row to put the newest recipe at the top (Row 2, just under headers)
        sheet.insert_row([timestamp, title, recipe], index=2)
    except Exception as e:
        print(f"Cloud Save Error: {e}")

def load_history_from_cloud():
    try:
        # Get everything including headers
        all_rows = sheet.get_all_values()
        
        # If there's only 1 row (just the headers) or 0 rows, it's empty
        if len(all_rows) <= 1:
            return "### 📜 History is empty\nStart cooking to see your recipes here!"
            
        history_list = []
        
        # We skip the first row (headers) and take the next 10 rows
        # Since we use insert_row(index=2), the newest are already at the top!
        for row in all_rows[1:11]: 
            # row[0] is Timestamp, row[1] is Dish, row[2] is Recipe
            time_st = row[0] if len(row) > 0 else ""
            title = row[1] if len(row) > 1 else "Untitled Dish"
            content = row[2] if len(row) > 2 else "No instructions found."
            
            history_list.append(f"### {title}\n*Generated on: {time_st}*\n\n{content}")
            
        return "\n\n---\n\n".join(history_list)
    except Exception as e:
        print(f"UI History Error: {e}")
        return "### ⚠️ System is warming up...\nIf this persists, check your Google Sheet connection."
# 4. The Generation Logic
def generate_full_meal(ingredients, dietary_type, creativity_val):
    try:
        # --- STEP 1: THE TITLE ---
        title_prompt = f"### Instruction: Invent a short 3-word name for a dish with: {ingredients}\n### Title:"
        title_inputs = tokenizer(title_prompt, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            title_outputs = model.generate(title_inputs["input_ids"], max_new_tokens=15, temperature=0.1)
        
        full_title_text = tokenizer.decode(title_outputs[0], skip_special_tokens=True)
        generated_title = full_title_text.split("### Title:")[-1].strip().split("\n")[0]

        # --- STEP 2: METHOD & DIET LOGIC ---
        diet_rule = f"Strictly adhere to the {dietary_type} diet." if dietary_type != "None" else "Provide a balanced recipe."
        
        # Decide the cooking method based on ingredients
        method = "stovetop" 
        ing_lower = ingredients.lower()
        if "flour" in ing_lower or "egg" in ing_lower or "yeast" in ing_lower:
            method = "baking"
        elif "oil" in ing_lower or "butter" in ing_lower or "ghee" in ing_lower:
            method = "pan-frying"

        # --- STEP 3: THE IMPROVED ONE-SHOT PROMPT ---
        recipe_prompt = f"""### System: 
You are a Professional Chef. Provide a {method} recipe. Follow the exact structure of the Example below.
Rules:
- 4-5 short, numbered steps only.
- No slashes (/) or repetitive words.
- Focus on the cooking process.
- Strictly follow ingredients list.
- Strictly follow diet rules if ingredients lies outside state- diet and ingredients does not match.

### Example:
Dish: Lentil Soup
Method: stovetop
Recipe:
1. Wash the lentils in cold water and drain.
2. Place lentils in a pot with water and bring to a boil.
3. Add salt and cumin, then simmer for 20 minutes.
4. Stir well and serve hot.

### Current Task:
Dish: {generated_title}
Method: {method}
Ingredients: {ingredients}
Rules: {diet_rule}

### Recipe Instructions:
1."""
        
        recipe_inputs = tokenizer(recipe_prompt, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            recipe_outputs = model.generate(
                recipe_inputs["input_ids"], 
                max_new_tokens=180, 
                temperature=0.1,         # Logic setting
                repetition_penalty=1.8,  # High penalty to stop loops
                do_sample=True,         
                pad_token_id=tokenizer.eos_token_id,
                attention_mask=recipe_inputs["attention_mask"]
            )
            
        full_text = tokenizer.decode(recipe_outputs[0], skip_special_tokens=True)

        # --- STEP 4: CLEANER & FORMATTER ---
        if "### Recipe Instructions:" in full_text:
            raw_recipe = full_text.split("### Recipe Instructions:")[-1].strip()
        else:
            raw_recipe = full_text.split("Recipe Instructions:")[-1].strip()

        # Split by period and force numbering
        sentences = raw_recipe.split(". ")
        cleaned_steps = []
        step_count = 1
        
        for sentence in sentences:
            sentence = sentence.strip()
            if len(sentence) > 8: 
                # Remove slashes and fix messy words
                sentence = sentence.replace("/", " or ")
                clean_sentence = sentence.lstrip("1234567890. ").strip()
                cleaned_steps.append(f"{step_count}. {clean_sentence}")
                step_count += 1
            if step_count > 6: 
                break

        recipe_only = "\n".join(cleaned_steps)
        if not recipe_only.endswith("."):
            recipe_only += "."

        return generated_title, recipe_only

    except Exception as e:
        return "Error", f"Kitchen Accident: {str(e)}"

# 5. Build the Web Interface
recipe_history = []

def ui_generate(ingredients, dietary, creativity):
    title, recipe = generate_full_meal(ingredients, dietary, creativity) 
    
    # Save to Cloud
    save_to_cloud(title, recipe)
    
    display_recipe = recipe
    if dietary != "None":
        display_recipe = f"**[{dietary} Option]**\n\n" + recipe
    
    # RE-FETCH the history from the Cloud to update the UI
    updated_history = load_history_from_cloud()
    
    return f"### {title}", display_recipe, updated_history

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 👨‍🍳 AI Recipe Chef")
    
    with gr.Sidebar():
        gr.Markdown("### ⚙️ Kitchen Settings")
        dietary_input = gr.Dropdown(
            ["None", "Vegan", "Vegetarian", "Gluten-Free", "Keto"], 
            label="Dietary Restriction", 
            value="None"
        )
        creativity_input = gr.Slider(
            0.1, 1.0, value=0.1, 
            label="Chef Creativity (Temperature)"
        )
        gr.Markdown("---")
        gr.Info("Running on 4-bit Quantized Qwen 2.5-3B")

    with gr.Row():
        with gr.Column(scale=1):
            ing_input = gr.Textbox(
                lines=5, 
                label="🛒 What's in your fridge?", 
                placeholder="Enter ingredients (e.g., chicken, onion, ginger...)"
            )
            submit_btn = gr.Button("Cook my meal! 🍳", variant="primary")

        with gr.Column(scale=2):
            title_output = gr.Markdown("### Your Dish Name will appear here")
            recipe_output = gr.Markdown("The step-by-step recipe will appear here.")

    with gr.Accordion("📜 Previous Recipes (Cloud History)", open=False):
        # This calls the cloud when the page loads
        history_box = gr.Markdown(load_history_from_cloud())

    submit_btn.click(
        fn=ui_generate, 
        inputs=[ing_input, dietary_input, creativity_input], 
        outputs=[title_output, recipe_output, history_box]
    )

demo.launch(share=True)

In [ ]:
# Run this in Kaggle to get ROUGE scores (Text Accuracy)
!pip install evaluate rouge_score

import evaluate
rouge = evaluate.load("rouge")

# Compare AI output to a Real Recipe
predictions = ["1. Wash rice. 2. Boil water."]
references = ["1. Wash the rice well. 2. Boil 2 cups of water."]

results = rouge.compute(predictions=predictions, references=references)
print(f"ROUGE Score: {results}")